# Prodigy_MultiParam_Study_PostFix — R2.4F (93 esperimenti)

**Obiettivo**: rifare lo studio multiparametrico Prodigy con il codice POST-FIX (4 bug strutturali risolti il 2026-06-03).

## Contesto

Il run R2.4 originale (90 esperimenti, in corso su Azure al momento del fix) usava codice con 4 bug strutturali documentati in [`document/BUGS_2026-06-03.md`](document/BUGS_2026-06-03.md):

1. **F5 sigmoid saturation** in `_decode_params` → 4/5 parametri IDM bloccati al random init (gradient ≈ 0)
2. **Xavier asymmetric bias** in `OutputLayer_LI` → determina deterministicamente quale bound viene saturato
3. **ALIF cascade dead output** in stacked layers (non rilevante per A1 baseline qui)
4. **Delay mask 1/max_delay penalty** → var(current) ridotta di max_delay

Conseguenza: TUTTI i ranking pregress (T30, P15, SW, R2.2, R2.4 highway) sono CORROTTI. La rete imparava solo `v0`, gli altri 4 parametri erano stuck.

**Verifica empirica post-fix (random init, 3 seeds)**:
- Saturation: **0%** su tutti i 5 canali (vs pre-fix 97% T, 96% s0)
- Spike rate: 6-10% (target [3%, 15%]) ✓
- Smoke locale 2 ep × 50 step: val_total = **0.213** (vs pregress floor 0.22 dopo 5700 step) → convergenza ~57× più veloce

## Setup

- **90 run Prodigy**: 3 LR ∈ {1.0, 0.5, 0.1} × 10 varianti × 3 scenari × 10 ep × 100 step
- **3 run AdamW baseline**: 1 per scenario, lr=1e-3 (per misurare il VALORE AGGIUNTO di Prodigy post-fix)
- **TOTALE: 93 run** (~15h Azure)
- Base canonical Prodigy: `betas=(0.9, 0.99)` (W1) + `use_bias_correction=True` (W3) + `safeguard_warmup=True` + `weight_decay=0.01`
- Arch: `baseline` (864 params, post-fix) — non più `BASELINE_BPTT_864p_PRE_EVENTPROP` perché ora la rete IMPARA con quella standard
- Push per-run dopo ogni esperimento
- Results: `results/Prodigy_Study/MultiParam_PostFix/<scenario>/<tag>/`
- Tag prefix: `R24F_` (F = Fixed) per separare archeologicamente dai 60 run pregress invalidati

## 10 varianti Prodigy (per LR fisso)

| ID | d_coef | d0 | growth_rate | scheduler | Logica |
|---|---:|---:|---:|---|---|
| V01 | 1.0 | 1e-6 | inf | none | Reference canonical |
| V02 | 0.5 | 1e-6 | inf | none | Brake leggero |
| V03 | 0.1 | 1e-6 | inf | none | Brake medio |
| V04 | 0.01 | 1e-6 | inf | none | Brake forte |
| V05 | 1.0 | 1e-6 | 1.02 | none | Growth cap smooth (paper) |
| V06 | 1.0 | 1e-6 | 1.01 | none | Growth cap stretto |
| V07 | 1.0 | 1e-4 | inf | none | d0 preheat (canonical kohya) |
| V08 | 1.0 | 1e-6 | inf | cosine_no_restart | Cosine decay |
| V09 | 0.1 | 1e-6 | 1.02 | cosine_no_restart | Combo brake+cap+cosine |
| V10 | 2.0 | 1e-6 | inf | none | d_coef=2 canonical kohya |

## 3 scenari training

1. **highway**: solo highway (cache F2 esistente)
2. **mixed**: 4 scenari (highway 40% + urban 30% + truck 20% + mixed 10%), no cut-in
3. **full**: SCENARIO_MIX default config + cut_in_ratio=0.2

## Tag naming

- Prodigy: `R24F_<scenario>_lr<L>_<V>` es. `R24F_highway_lr1.0_V01`
- AdamW baseline: `R24F_<scenario>_adamw_lr1e-3`

## Note operative

- **Cell 4 (PRE-FLIGHT SANITY)**: smoke 1 ep × 5 step + verifica saturation < 5% + val_total numerico, ABORT se fallisce
- **SKIP_IF_EXISTS** robusto: i restart non rifanno run completate
- **Differenze rispetto R2.4 originale**: arch `baseline` post-fix (era `BASELINE_BPTT_864p_PRE_EVENTPROP`) + dir/prefix nuovi + AdamW comparison + pre-flight sanity

**Reference**: `PRODIGY_DEEP_STUDY.md` parte 1+2+3, `document/BUGS_2026-06-03.md`.

In [ ]:
# Cell 1 -- Bootstrap + ENV check + verifica FIX applicati
import sys, os, subprocess, inspect
import importlib.util as _imu

# Install deps (idempotent)
for pkg in ['prodigyopt', 'pandas', 'matplotlib']:
    if _imu.find_spec(pkg) is None:
        print(f'  installing {pkg}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

# Verifica file critici
for f in ['train.py', 'core/network.py', 'core/eventprop.py', 'config.py', 'data/generator.py',
          'document/BUGS_2026-06-03.md']:
    assert os.path.isfile(f), f'MISSING: {f}'
    print(f'  [OK] {f}')

# ====== VERIFICA STATICA FIX BUGS_2026-06-03 ======
print('\n--- Verifica FIX applicati (statica) ---')
with open('core/network.py', encoding='utf-8') as fp:
    net_src = fp.read()
assert 'raw / self.decode_scale' not in net_src, 'FIX#1 NON applicato (decode_scale ancora usato)'
assert 'FIX-BUG-1' in net_src, 'FIX#1 marker mancante'
assert 'FIX-BUG-2' in net_src, 'FIX#2 marker mancante (OutputLayer_LI row-mean)'
assert 'FIX-BUG-3' in net_src, 'FIX#3 marker mancante (cascade base_threshold)'
assert 'FIX-BUG-4' in net_src, 'FIX#4 marker mancante (delay mask compensation)'
with open('core/eventprop.py', encoding='utf-8') as fp:
    ep_src = fp.read()
assert 'FIX-BUG-2' in ep_src and 'FIX-BUG-4' in ep_src, 'FIX EVPROP mancanti'
print('  [OK] FIX#1 (decode_scale rimosso) presente in core/network.py')
print('  [OK] FIX#2 (LI row-mean) presente in core/network.py + core/eventprop.py')
print('  [OK] FIX#3 (cascade thr=1.0) presente in core/network.py')
print('  [OK] FIX#4 (delay mask sqrt) presente in core/network.py + core/eventprop.py')

# Prodigy API
from prodigyopt import Prodigy
sig = inspect.signature(Prodigy.__init__)
for p in ['safeguard_warmup','growth_rate','d_coef','d0','betas','use_bias_correction','weight_decay']:
    assert p in sig.parameters, f'MISSING Prodigy param: {p}'
print('\n[OK] Prodigy API: 7 param richiesti presenti')

# train.py CLI flags
help_txt = subprocess.run([sys.executable, 'train.py', '--help'],
                           capture_output=True, text=True, encoding='utf-8',
                           errors='replace').stdout
for flag in ['--prodigy_safeguard_warmup','--prodigy_growth_rate','--prodigy_d_coef',
             '--prodigy_betas','--prodigy_use_bias_correction','--prodigy_d0',
             '--prodigy_weight_decay']:
    assert flag in help_txt, f'MISSING train.py flag: {flag}'
assert 'cosine_no_restart' in help_txt, 'MISSING scheduler cosine_no_restart'
print('[OK] train.py CLI flags Prodigy + scheduler completi')

# git config
br = subprocess.run(['git','branch','--show-current'], capture_output=True, text=True).stdout.strip()
print(f'\n[GIT] branch corrente: {br}')
assert br == 'Prodigy_Deep_Study', f'Wrong branch: {br} (expected Prodigy_Deep_Study)'
print('[OK] su branch Prodigy_Deep_Study, push per-run abilitato')

# Tag pre-fix esiste?
tags = subprocess.run(['git','tag','--list'], capture_output=True, text=True).stdout
if 'pre_bug_fix_2026-06-03' in tags:
    print('[OK] tag pre_bug_fix_2026-06-03 presente (per rollback se necessario)')
else:
    print('[WARN] tag pre_bug_fix_2026-06-03 NON trovato')

print('\nENV check passed.')

In [ ]:
# Cell 2 -- Generate 93 EXPERIMENTS list (90 Prodigy + 3 AdamW baseline)
import itertools

VARIANTS = [
    # ID, d_coef, d0, growth_rate, scheduler, desc
    ('V01', 1.0,   1e-6, 'inf',  'none',              'Reference canonical (W1+W3 only)'),
    ('V02', 0.5,   1e-6, 'inf',  'none',              'Brake leggero (d_coef=0.5)'),
    ('V03', 0.1,   1e-6, 'inf',  'none',              'Brake medio (d_coef=0.1)'),
    ('V04', 0.01,  1e-6, 'inf',  'none',              'Brake forte (d_coef=0.01)'),
    ('V05', 1.0,   1e-6, '1.02', 'none',              'Growth_rate cap smooth (paper 1.02)'),
    ('V06', 1.0,   1e-6, '1.01', 'none',              'Growth_rate cap stretto (1.01)'),
    ('V07', 1.0,   1e-4, 'inf',  'none',              'd0 preheat alto (canonical kohya)'),
    ('V08', 1.0,   1e-6, 'inf',  'cosine_no_restart', 'Cosine sched (decay finale)'),
    ('V09', 0.1,   1e-6, '1.02', 'cosine_no_restart', 'Combo brake + cap + cosine'),
    ('V10', 2.0,   1e-6, 'inf',  'none',              'd_coef=2 canonical kohya'),
]
LRS = [1.0, 0.5, 0.1]
SCENARIOS = [
    ('highway', 'highway',                                        0.0, 'data/cache_1500_highway_cut0.0_ou0.0.pt'),
    ('mixed',   'highway:0.4,urban:0.3,truck:0.2,mixed:0.1',        0.0, 'data/cache_1500_mixed_cut0.0_ou0.0.pt'),
    ('full',    'default',                                         0.2, 'data/cache_1500_full_cut0.2_ou0.0.pt'),
]

EXPERIMENTS = []
# 90 Prodigy run
for (scn_name, scn_mix, cut_in, cache_path), lr, (vid, dcoef, d0, growth, sched, desc) in \
        itertools.product(SCENARIOS, LRS, VARIANTS):
    tag = f'R24F_{scn_name}_lr{lr}_{vid}'
    EXPERIMENTS.append({
        'tag': tag, 'scenario': scn_name, 'scenario_mix': scn_mix, 'cut_in_ratio': cut_in,
        'cache_path': cache_path, 'optimizer': 'prodigy', 'lr': lr,
        'variant_id': vid, 'd_coef': dcoef, 'd0': d0,
        'growth_rate': growth, 'scheduler': sched, 'desc': desc,
    })

# 3 AdamW baseline (1 per scenario)
for scn_name, scn_mix, cut_in, cache_path in SCENARIOS:
    tag = f'R24F_{scn_name}_adamw_lr1e-3'
    EXPERIMENTS.append({
        'tag': tag, 'scenario': scn_name, 'scenario_mix': scn_mix, 'cut_in_ratio': cut_in,
        'cache_path': cache_path, 'optimizer': 'adamw', 'lr': 1e-3,
        'variant_id': 'ADAMW_REF', 'd_coef': None, 'd0': None,
        'growth_rate': None, 'scheduler': 'plateau',
        'desc': 'AdamW baseline lr=1e-3 (riferimento per valore aggiunto Prodigy)',
    })

print(f'Total experiments: {len(EXPERIMENTS)} (atteso 93 = 90 Prodigy + 3 AdamW)')
assert len(EXPERIMENTS) == 93, f'Expected 93, got {len(EXPERIMENTS)}'
for s in ['highway','mixed','full']:
    n_prod = sum(1 for e in EXPERIMENTS if e['scenario']==s and e['optimizer']=='prodigy')
    n_adam = sum(1 for e in EXPERIMENTS if e['scenario']==s and e['optimizer']=='adamw')
    print(f'  {s}: {n_prod} Prodigy + {n_adam} AdamW')
print(f'\nFirst Prodigy: {EXPERIMENTS[0]["tag"]}')
print(f'Last  Prodigy: {EXPERIMENTS[89]["tag"]}')
print(f'AdamW baselines: {[e["tag"] for e in EXPERIMENTS[-3:]]}')

In [ ]:
# Cell 3 -- Cache check + AUTOGEN per scenari mancanti (riusa cache R2.4 originale)
import os, time
import torch

for scn_name, scn_mix, cut_in, cache_path in SCENARIOS:
    if os.path.isfile(cache_path):
        sz = os.path.getsize(cache_path) / 1024 / 1024
        print(f'[OK] {scn_name}: {cache_path} ({sz:.1f} MB)')
        continue
    print(f'\n[AUTOGEN] {scn_name}: {cache_path} mancante, genero...')
    print(f'  scenario_mix={scn_mix} cut_in_ratio={cut_in}')
    from data.generator import generate_dataset, parse_scenario_mix
    from config import SEED
    mix_dict = parse_scenario_mix(scn_mix)
    t0 = time.time()
    train_d = generate_dataset(1500, base_seed=SEED, scenario_mix=mix_dict,
                                cut_in_ratio=cut_in, noise_scale=0.0)
    val_d   = generate_dataset(300,  base_seed=SEED+1, scenario_mix=mix_dict,
                                cut_in_ratio=cut_in, noise_scale=0.0)
    torch.save({'train': train_d, 'val': val_d, 'seed': SEED}, cache_path)
    sz = os.path.getsize(cache_path) / 1024 / 1024
    print(f'  [SAVED] {cache_path} ({sz:.1f} MB) in {time.time()-t0:.0f}s')
print('\nAll caches ready.')

In [ ]:
# Cell 4 -- PRE-FLIGHT SANITY (NEW post-fix)
# Verifica EMPIRICA che i fix funzionino prima di lanciare 15h di compute.
# Costo: ~30s locali. Risparmio: evitare di scoprire dopo 15h che il rerun era inutile.
import torch, inspect, sys, numpy as np
sys.path.insert(0, '.')
from core.network import build_model

print('=== PRE-FLIGHT 1: Static fix check ===')
torch.manual_seed(42)
m = build_model('baseline')
src = inspect.getsource(m._decode_params)
assert 'raw / self.decode_scale' not in src, 'FIX#1 NON applicato'
assert 'torch.sigmoid(raw)' in src, 'FIX#1 forma errata'
fc = m.layer_out.fc_weight
row_mean = fc.mean(dim=1).abs().max().item()
assert row_mean < 1e-6, f'FIX#2 NON applicato (row_mean={row_mean})'
n_params = sum(p.numel() for p in m.parameters())
assert n_params == 864, f'Param count drift: {n_params} != 864'
print(f'  FIX#1 (_decode_params no decode_scale): OK')
print(f'  FIX#2 (LI row_mean = {row_mean:.2e}): OK')
print(f'  Param count A1 = {n_params}: OK')

print('\n=== PRE-FLIGHT 2: Empirical saturation (random init, 3 seeds) ===')
cache = torch.load('data/cache_1500_highway_cut0.0_ou0.0.pt', weights_only=False)
xs = []
for s in cache['train'][:20]:
    x = s['x']
    if isinstance(x, np.ndarray): x = torch.from_numpy(x).float()
    xs.append(x[:30])
x = torch.stack(xs)

sat_max_all = []
grad_min_all = []
sr_all = []
for seed in [42, 123, 7]:
    torch.manual_seed(seed)
    m = build_model('baseline')
    m.train()
    steps, sp = m.forward_sequence_with_stats(x)
    plo, phi = m.param_lo.view(1,1,-1), m.param_hi.view(1,1,-1)
    eps = 0.01*(phi-plo)
    sat = ((steps < plo+eps).float() + (steps > phi-eps).float()).mean(dim=(0,1)).cpu().numpy()
    sat_max_all.append(sat.max())
    sr_all.append(sp.mean().item())
    loss = ((steps - (plo+phi)/2)**2).mean()
    loss.backward()
    grad_min_all.append(m.layer_out.fc_weight.grad.abs().mean(dim=1).min().item())

sat_max = max(sat_max_all)
grad_min = min(grad_min_all)
sr_mean = float(np.mean(sr_all))
print(f'  Saturation max (any seed, any param): {sat_max*100:.2f}% (target <5%)')
print(f'  Gradient min (any seed, any LI ch):   {grad_min:.5f} (target >1e-5)')
print(f'  Spike rate mean:                       {sr_mean:.3f} (target [0.03, 0.20])')

assert sat_max < 0.05, f'FAIL: saturation {sat_max*100:.1f}% > 5%. ABORT sweep — fix non efficace.'
assert grad_min > 1e-5, f'FAIL: gradient {grad_min:.2e} ~0. ABORT sweep — qualche canale ancora bloccato.'
assert 0.03 <= sr_mean <= 0.20, f'FAIL: spike rate {sr_mean:.3f} fuori range. ABORT sweep.'

print('\n=== PRE-FLIGHT 3: Smoke 1 ep × 5 step (training loop integro) ===')
import subprocess as sp
r = sp.run([sys.executable, 'train.py',
            '--training_method', 'baseline',
            '--epochs', '1', '--max_steps_per_epoch', '5',
            '--batch_size', '4', '--seq_len', '30',
            '--data_cache', 'data/cache_1500_highway_cut0.0_ou0.0.pt',
            '--n_train', '16', '--n_val', '8',
            '--tag', '_PREFLIGHT_R24F'],
           capture_output=True, text=True, encoding='utf-8', errors='replace')
assert r.returncode == 0, f'Smoke train.py FAILED rc={r.returncode}\n{r.stderr[-500:]}'
import os, shutil
log = 'checkpoints/_PREFLIGHT_R24F/training_log.csv'
assert os.path.isfile(log), 'training_log.csv mancante'
shutil.rmtree('checkpoints/_PREFLIGHT_R24F', ignore_errors=True)
print('  Smoke train.py 1 ep × 5 step: exit 0, log generato')

print('\n[OK] Pre-flight sanity superato. Procedere con sweep 93 esperimenti.')

In [ ]:
# Cell 5 -- RUN sweep 93 esperimenti con PUSH per-run (pattern T30 BEST)
import subprocess, sys, time, os, shutil, glob, datetime
import pandas as pd

SKIP_IF_EXISTS = True   # RESUME safe
RESULTS_DIR = 'results/Prodigy_Study/MultiParam_PostFix'
os.makedirs(RESULTS_DIR, exist_ok=True)

_TMP_MSG = '/tmp/r24f_msg.txt' if os.path.isdir('/tmp') else 'r24f_msg.txt'

def _build_cli(e):
    common = [sys.executable, 'train.py',
        '--training_method', 'baseline',
        '--epochs', '10', '--max_steps_per_epoch', '100',
        '--batch_size', '8', '--val_batch_size', '32', '--seq_len', '50',
        '--cf_hidden_size', '32', '--cf_rank', '8',
        '--lambda_data', '1.0', '--lambda_phys', '0.1', '--lambda_ou', '0.05',
        '--lambda_bc', '1.0', '--lambda_sr', '0.5',
        '--scenario_mix', e['scenario_mix'],
        '--cut_in_ratio', str(e['cut_in_ratio']),
        '--noise_scale', '0.0', '--po2_enabled', '1',
        '--n_train', '1500', '--n_val', '300',
        '--max_inf_streak', '99999', '--early_stop_patience', '0',
        '--data_cache', e['cache_path'],
        '--tag', e['tag']]
    if e['optimizer'] == 'prodigy':
        common += [
            '--optimizer', 'prodigy',
            '--lr', str(e['lr']), '--max_lr', str(e['lr']),
            '--scheduler', e['scheduler'],
            '--prodigy_betas', '0.9,0.99',
            '--prodigy_d_coef', str(e['d_coef']),
            '--prodigy_d0', str(e['d0']),
            '--prodigy_weight_decay', '0.01',
            '--prodigy_use_bias_correction', '1',
            '--prodigy_safeguard_warmup', '1',
            '--prodigy_growth_rate', str(e['growth_rate'])]
    else:  # adamw baseline
        common += [
            '--optimizer', 'adamw',
            '--lr', str(e['lr']), '--max_lr', str(e['lr']),
            '--scheduler', e['scheduler']]
    return common

def _push_run(e):
    """Move checkpoints/<tag>/ -> results/Prodigy_Study/MultiParam_PostFix/<scenario>/<tag>/ + git push."""
    tag = e['tag']
    src = f'checkpoints/{tag}'
    dst_parent = f'{RESULTS_DIR}/{e["scenario"]}'
    dst = f'{dst_parent}/{tag}'
    if not os.path.isdir(src):
        print(f'   [WARN push] {src} mancante, skip')
        return False
    os.makedirs(dst_parent, exist_ok=True)
    if os.path.isdir(dst): shutil.rmtree(dst)
    os.makedirs(f'{dst}/plots', exist_ok=True)
    for f in glob.glob(f'{src}/*.csv') + glob.glob(f'{src}/*.json'):
        shutil.copy2(f, dst)
    for f in glob.glob(f'{src}/plots/*.png'):
        shutil.copy2(f, f'{dst}/plots/')

    val_str = ''
    if os.path.isfile(f'{dst}/training_log.csv'):
        try:
            edf = pd.read_csv(f'{dst}/training_log.csv')
            if len(edf) > 0:
                bi = int(edf.val_total.idxmin())
                val_str = f'best val_total={edf.val_total.min():.4f} val_data={edf.val_data.iloc[bi]:.4f} (E{bi+1}/{len(edf)})'
        except Exception as ex:
            val_str = f'(log read failed: {ex})'

    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    if e['optimizer'] == 'prodigy':
        opt_str = f'lr={e["lr"]} d_coef={e["d_coef"]} d0={e["d0"]} growth={e["growth_rate"]} sched={e["scheduler"]}'
    else:
        opt_str = f'adamw lr={e["lr"]} sched={e["scheduler"]}'
    msg = (
        f'results (R2.4F Prodigy MultiParam PostFix): {tag} ({ts})\n\n'
        f'{val_str}\n'
        f'scenario={e["scenario"]} {opt_str}\n'
        f'Branch Prodigy_Deep_Study | post BUGS_2026-06-03 fix\n'
    )
    with open(_TMP_MSG, 'w', encoding='utf-8') as fp:
        fp.write(msg)
    try:
        subprocess.run(['git','add',dst], check=True, capture_output=True)
        r = subprocess.run(['git','commit','-F',_TMP_MSG], capture_output=True, text=True)
        if r.returncode != 0:
            if 'nothing to commit' in r.stdout or 'nothing to commit' in r.stderr:
                print(f'   [push] nothing to commit')
                return True
            print(f'   [push] commit failed rc={r.returncode}: {r.stderr[-300:]}')
            return False
        subprocess.run(['git','pull','--no-rebase','--no-edit','origin','Prodigy_Deep_Study'],
                       capture_output=True, text=True)
        r2 = subprocess.run(['git','push','origin','Prodigy_Deep_Study'], capture_output=True, text=True)
        if r2.returncode != 0:
            print(f'   [push] push failed rc={r2.returncode}: {r2.stderr[-300:]}')
            return False
        print(f'   [push] OK')
        return True
    finally:
        try: os.remove(_TMP_MSG)
        except: pass

# Esecuzione
run_results = []
t_start = time.time()
total = len(EXPERIMENTS)
for i, e in enumerate(EXPERIMENTS, 1):
    tag = e['tag']
    dst_log = f'{RESULTS_DIR}/{e["scenario"]}/{tag}/training_log.csv'
    if SKIP_IF_EXISTS and os.path.isfile(dst_log):
        try:
            edf = pd.read_csv(dst_log)
            v_str = f'val_total={edf.val_total.min():.4f}'
        except Exception:
            v_str = 'unreadable'
        print(f'\n[{i}/{total}] [SKIP_EXIST] {tag}: {v_str}')
        run_results.append({'tag': tag, 'status':'skipped'})
        continue
    print(f'\n{"="*78}\n[{i}/{total}] {tag}: {e["desc"]}\n   scn={e["scenario"]} opt={e["optimizer"]} lr={e["lr"]}\n{"="*78}')
    t0 = time.time()
    r = subprocess.run(_build_cli(e), capture_output=False)
    el = time.time() - t0
    status = 'ok' if r.returncode == 0 else f'fail({r.returncode})'
    el_tot = time.time() - t_start
    done_now = sum(1 for x in run_results if x['status']!='skipped') + 1
    eta_min = (el_tot / max(done_now,1)) * (total - i) / 60
    print(f'\n[{i}/{total}] {tag} -> {status}  ({el/60:.1f}min)  ETA={eta_min:.0f}min')
    print(f'   pushing {tag}...')
    pushed = _push_run(e)
    run_results.append({'tag': tag, 'status': status, 'pushed': pushed, 'elapsed': el})

print(f'\n{"="*78}\nSWEEP DONE in {(time.time()-t_start)/60:.0f}min')
for r in run_results[-10:]:
    if r['status'] != 'skipped':
        print(f"   {r['tag']:<45} {r['status']:<10} push={r.get('pushed','N/A')}")

In [ ]:
# Cell 6 -- Aggregate analysis: leggi tutti i csv -> DataFrame unico
import pandas as pd, os, math
RESULTS_DIR = 'results/Prodigy_Study/MultiParam_PostFix'

rows = []
for e in EXPERIMENTS:
    tag = e['tag']
    base = f'{RESULTS_DIR}/{e["scenario"]}/{tag}'
    log = f'{base}/training_log.csv'
    blog = f'{base}/training_batch_log.csv'
    row_base = {k: e.get(k) for k in ['tag','scenario','optimizer','lr','variant_id','d_coef','d0','growth_rate','scheduler','desc']}
    if not os.path.isfile(log):
        rows.append({**row_base, 'status':'MISSING'})
        continue
    df = pd.read_csv(log)
    bdf = pd.read_csv(blog) if os.path.isfile(blog) else None
    if len(df) == 0:
        rows.append({**row_base, 'status':'EMPTY'})
        continue
    vts = df['val_total'].values
    bi = int(df['val_total'].idxmin())
    if bdf is not None and 'gn_total_preclip' in bdf.columns:
        gn = bdf['gn_total_preclip']
        n_inf = int(gn.apply(lambda x: math.isinf(x) if isinstance(x,(int,float)) else False).sum())
        n_huge = int((gn > 1e6).sum())
        gn_explosion_pct = (n_inf + n_huge) / len(bdf) * 100
    else:
        gn_explosion_pct = float('nan')
    if bdf is not None and 'prodigy_lr_eff' in bdf.columns:
        lr_eff_start = float(bdf['prodigy_lr_eff'].iloc[0]) if not bdf['prodigy_lr_eff'].isna().all() else float('nan')
        lr_eff_max = float(bdf['prodigy_lr_eff'].max()) if not bdf['prodigy_lr_eff'].isna().all() else float('nan')
        lr_eff_end = float(bdf['prodigy_lr_eff'].iloc[-1]) if not bdf['prodigy_lr_eff'].isna().all() else float('nan')
    else:
        lr_eff_start=lr_eff_max=lr_eff_end=float('nan')
    row = {**row_base,
        'status':'OK', 'n_ep': len(df),
        'val_total_best': float(vts[bi]), 'best_ep': bi+1,
        'val_total_last': float(vts[-1]),
        'val_data_at_best': float(df['val_data'].iloc[bi]),
        'spike_rate_avg': float(df['spike_rate'].mean()),
        'gn_explosion_pct': gn_explosion_pct,
        'lr_eff_start': lr_eff_start, 'lr_eff_max': lr_eff_max, 'lr_eff_end': lr_eff_end,
    }
    rows.append(row)

df_all = pd.DataFrame(rows)
n_ok = (df_all.status=='OK').sum()
n_missing = (df_all.status=='MISSING').sum()
print(f'Total runs: {len(df_all)} (OK: {n_ok}, MISSING: {n_missing})')
os.makedirs(RESULTS_DIR, exist_ok=True)
df_all.to_csv(f'{RESULTS_DIR}/_aggregate.csv', index=False)
print(f'[saved] {RESULTS_DIR}/_aggregate.csv')
from IPython.display import display
print('\nTOP 15 best val_total_best (post-fix):')
display(df_all[df_all.status=='OK'].sort_values('val_total_best').head(15).round(4))
print('\nAdamW baselines:')
display(df_all[(df_all.status=='OK') & (df_all.optimizer=='adamw')].round(4))

In [ ]:
# Cell 7 -- PARETO plots per scenario (val_total vs grad_explosion / lr_eff / spike_rate)
# + linea orizzontale = AdamW baseline per quel scenario (per misurare valore aggiunto Prodigy)
import matplotlib.pyplot as plt, os

ok = df_all[df_all.status=='OK'].copy()
ok_prodigy = ok[ok.optimizer=='prodigy'].copy()
ok_adamw   = ok[ok.optimizer=='adamw'].copy()
os.makedirs('opt_plots/R24F_pareto', exist_ok=True)

for scn in ['highway','mixed','full']:
    sub = ok_prodigy[ok_prodigy.scenario==scn].copy()
    adamw_ref = ok_adamw[ok_adamw.scenario==scn]
    adamw_val = float(adamw_ref.val_total_best.iloc[0]) if len(adamw_ref) > 0 else None
    if len(sub) == 0:
        print(f'[skip plot] {scn}: no Prodigy OK runs')
        continue
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    # Panel 1: val_total vs grad_explosion_pct
    for lr_v in sub.lr.unique():
        ss = sub[sub.lr==lr_v]
        axes[0].scatter(ss.gn_explosion_pct, ss.val_total_best, s=80, alpha=0.7, label=f'Prodigy lr={lr_v}')
        for _, r in ss.iterrows():
            axes[0].annotate(r.variant_id, (r.gn_explosion_pct, r.val_total_best), fontsize=7, alpha=0.7)
    if adamw_val:
        axes[0].axhline(adamw_val, color='red', ls='--', lw=1.5, label=f'AdamW baseline ({adamw_val:.4f})')
    axes[0].set_xlabel('gradient explosion %  (lower=better)')
    axes[0].set_ylabel('val_total best  (lower=better)')
    axes[0].set_title(f'{scn}: Pareto val vs grad_explosion')
    axes[0].grid(alpha=0.3); axes[0].legend(fontsize=8)
    # Panel 2: val_total vs lr_eff_max
    for lr_v in sub.lr.unique():
        ss = sub[sub.lr==lr_v]
        axes[1].scatter(ss.lr_eff_max, ss.val_total_best, s=80, alpha=0.7, label=f'Prodigy lr={lr_v}')
        for _, r in ss.iterrows():
            axes[1].annotate(r.variant_id, (r.lr_eff_max, r.val_total_best), fontsize=7, alpha=0.7)
    if adamw_val:
        axes[1].axhline(adamw_val, color='red', ls='--', lw=1.5, label=f'AdamW baseline')
    axes[1].set_xscale('log')
    axes[1].set_xlabel('lr_eff_max (log, target ~2e-3 stable)')
    axes[1].set_ylabel('val_total best')
    axes[1].set_title(f'{scn}: Pareto val vs lr_eff_max')
    axes[1].grid(alpha=0.3); axes[1].legend(fontsize=8)
    # Panel 3: val_total vs spike_rate
    for lr_v in sub.lr.unique():
        ss = sub[sub.lr==lr_v]
        axes[2].scatter(ss.spike_rate_avg*100, ss.val_total_best, s=80, alpha=0.7, label=f'Prodigy lr={lr_v}')
        for _, r in ss.iterrows():
            axes[2].annotate(r.variant_id, (r.spike_rate_avg*100, r.val_total_best), fontsize=7, alpha=0.7)
    if adamw_val:
        axes[2].axhline(adamw_val, color='red', ls='--', lw=1.5, label=f'AdamW baseline')
    axes[2].axvspan(15, 20, alpha=0.15, color='green', label='target FPGA 15-20%')
    axes[2].set_xlabel('spike rate avg %')
    axes[2].set_ylabel('val_total best')
    axes[2].set_title(f'{scn}: Pareto val vs spike_rate')
    axes[2].grid(alpha=0.3); axes[2].legend(fontsize=8)
    fig.suptitle(f'R2.4F Prodigy MultiParam PostFix -- scenario={scn} (30 Prodigy + 1 AdamW)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    out = f'opt_plots/R24F_pareto/pareto_{scn}.png'
    fig.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'[saved] {out}')

In [ ]:
# Cell 8 -- TOP/WORST/stats per scenario + valore aggiunto Prodigy vs AdamW
from IPython.display import display, Markdown

for scn in ['highway','mixed','full']:
    sub = ok[ok.scenario==scn].sort_values('val_total_best')
    if len(sub) == 0: continue
    display(Markdown(f'## Scenario: {scn} ({len(sub)} run OK)'))
    cols_show = ['tag','optimizer','lr','variant_id','d_coef','growth_rate','scheduler',
                 'val_total_best','val_data_at_best','best_ep','spike_rate_avg',
                 'gn_explosion_pct','lr_eff_max']
    display(Markdown('### TOP 5 (best val_total)'))
    display(sub.head(5)[cols_show].round(4))
    display(Markdown('### AdamW baseline'))
    display(sub[sub.optimizer=='adamw'][cols_show].round(4))
    display(Markdown('### WORST 5'))
    display(sub.tail(5)[cols_show].round(4))
    # Delta vs AdamW: quanti Prodigy battono AdamW?
    ad = sub[sub.optimizer=='adamw']
    if len(ad) > 0:
        adamw_val = float(ad.val_total_best.iloc[0])
        n_better = (sub[sub.optimizer=='prodigy'].val_total_best < adamw_val).sum()
        n_prod_total = (sub.optimizer=='prodigy').sum()
        display(Markdown(f'### Valore aggiunto Prodigy: **{n_better}/{n_prod_total}** Prodigy battono AdamW baseline (val_total < {adamw_val:.4f})'))
    display(Markdown('### Stats per LR (Prodigy only)'))
    stats = sub[sub.optimizer=='prodigy'].groupby('lr').agg(
        n=('val_total_best','count'),
        val_total_min=('val_total_best','min'),
        val_total_mean=('val_total_best','mean'),
        val_total_max=('val_total_best','max'),
        gn_expl_mean=('gn_explosion_pct','mean'),
        lr_eff_max_median=('lr_eff_max','median'),
    ).round(4)
    display(stats)

In [ ]:
# Cell 9 -- Comparison PRE-FIX vs POST-FIX (se aggregate pre-fix esiste)
# Carica i due aggregate.csv e mostra delta side-by-side per ogni (scenario, lr, variant_id)
import pandas as pd, os
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PRE = 'results/Prodigy_Study/MultiParam/_aggregate.csv'
POST = 'results/Prodigy_Study/MultiParam_PostFix/_aggregate.csv'

if not os.path.isfile(PRE):
    display(Markdown('### Pre-fix aggregate NOT FOUND — skip comparison'))
    print(f'   Path tentato: {PRE}')
    print('   Esegui il vecchio notebook Prodigy_MultiParam_Study.ipynb cell 5 se vuoi rigenerare.')
elif not os.path.isfile(POST):
    display(Markdown('### Post-fix aggregate NOT YET GENERATED — esegui Cell 6 prima'))
else:
    df_pre = pd.read_csv(PRE)
    df_post = pd.read_csv(POST)
    # Match by (scenario, lr, variant_id) per le 90 Prodigy run
    df_pre_p = df_pre[df_pre.status=='OK'].copy() if 'status' in df_pre.columns else df_pre.copy()
    df_post_p = df_post[(df_post.status=='OK') & (df_post.optimizer=='prodigy')].copy()
    # Key: scenario + lr + variant_id
    df_pre_p['key'] = df_pre_p.scenario.astype(str) + '|' + df_pre_p.lr.astype(str) + '|' + df_pre_p.variant_id.astype(str)
    df_post_p['key'] = df_post_p.scenario.astype(str) + '|' + df_post_p.lr.astype(str) + '|' + df_post_p.variant_id.astype(str)
    merged = df_pre_p[['key','scenario','lr','variant_id','val_total_best','gn_explosion_pct','spike_rate_avg']].merge(
        df_post_p[['key','val_total_best','gn_explosion_pct','spike_rate_avg']],
        on='key', suffixes=('_pre','_post'), how='inner')
    merged['delta_val'] = merged.val_total_best_post - merged.val_total_best_pre
    merged['ratio_val'] = merged.val_total_best_post / merged.val_total_best_pre
    n_match = len(merged)
    n_improved = (merged.delta_val < 0).sum()
    display(Markdown(f'## Comparison Pre-fix vs Post-fix ({n_match} run matched)\n\n'
                     f'- **Improved** (delta_val < 0): **{n_improved}/{n_match}** ({100*n_improved/n_match:.0f}%)\n'
                     f'- Mean delta_val: **{merged.delta_val.mean():+.4f}**\n'
                     f'- Mean ratio post/pre: **{merged.ratio_val.mean():.3f}**'))
    # Scatter pre vs post
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for scn in ['highway','mixed','full']:
        ss = merged[merged.scenario==scn]
        axes[0].scatter(ss.val_total_best_pre, ss.val_total_best_post, s=60, alpha=0.7, label=scn)
    lim = [merged[['val_total_best_pre','val_total_best_post']].values.min()*0.9,
           merged[['val_total_best_pre','val_total_best_post']].values.max()*1.05]
    axes[0].plot(lim, lim, 'k--', alpha=0.5, label='y=x (no change)')
    axes[0].set_xlabel('val_total_best PRE-fix')
    axes[0].set_ylabel('val_total_best POST-fix')
    axes[0].set_title('Pre vs Post fix (sotto y=x = miglioramento)')
    axes[0].grid(alpha=0.3); axes[0].legend()
    # Histogram delta
    axes[1].hist(merged.delta_val, bins=30, edgecolor='black', alpha=0.7)
    axes[1].axvline(0, color='red', ls='--', label='no change')
    axes[1].axvline(merged.delta_val.mean(), color='green', ls='--', label=f'mean={merged.delta_val.mean():+.4f}')
    axes[1].set_xlabel('delta_val (post - pre)')
    axes[1].set_ylabel('# run')
    axes[1].set_title(f'Distribuzione delta_val ({n_match} matched runs)')
    axes[1].grid(alpha=0.3); axes[1].legend()
    plt.tight_layout()
    os.makedirs('opt_plots/R24F_pareto', exist_ok=True)
    fig.savefig('opt_plots/R24F_pareto/comparison_pre_vs_post.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('\nTOP 10 improvements (delta_val piu negativo = piu migliorato):')
    display(merged.nsmallest(10, 'delta_val')[['scenario','lr','variant_id','val_total_best_pre','val_total_best_post','delta_val','ratio_val']].round(4))
    print('\nTOP 10 regressions (delta_val piu positivo = peggiorato):')
    display(merged.nlargest(10, 'delta_val')[['scenario','lr','variant_id','val_total_best_pre','val_total_best_post','delta_val','ratio_val']].round(4))